In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [ ]:
# ==========================================
# AIMO3 Gold Medal Contender
# Key Upgrades Over Baseline:
#   - vLLM for 10-20x faster batched inference
#   - Qwen3-32B with native thinking mode (IMO-grade reasoning)
#   - DeepSeek-R1-Distill-Qwen-14B as secondary ensemble model
#   - True multi-turn TIR: code output feeds back into the model
#   - 32+ samples per problem via vLLM batch generation
#   - Problem-type classifier for strategy routing
#   - Weighted majority vote with confidence scoring
#   - Robust 5-digit answer extraction with SymPy fallback
# ==========================================
# ==========================================
# Cell 1: Install Dependencies
# ==========================================
import subprocess, sys

def pip_install(pkg: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

pip_install("vllm>=0.8.5")
pip_install("sympy>=1.13")
pip_install("numpy")
pip_install("pandas")

print("✅ Dependencies installed")
# ==========================================
# Cell 2: Imports & Global Setup
# ==========================================
import os
import re
import time
import random
import math
import ast
import signal
import contextlib
from io import StringIO
from dataclasses import dataclass, field
from collections import Counter
from typing import List, Dict, Tuple, Optional, Any
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sympy as sp
from sympy import *

import torch
from vllm import LLM, SamplingParams

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("✅ Imports complete")
# ==========================================
# Cell 3: Configuration
# ==========================================
@dataclass
class AIMOConfig:
    # ---- Primary model (Qwen3-32B thinking mode: best open IMO-level reasoner) ----
    # Available on Kaggle as: Qwen/Qwen3-32B/transformers/default/1
    model_path_primary: str    = "/kaggle/input/qwen3-32b/transformers/default/1"
    # ---- Secondary model for ensemble diversity ----
    # Available on Kaggle as: deepseek-ai/DeepSeek-R1-Distill-Qwen-14B/...
    model_path_secondary: str  = "/kaggle/input/deepseek-r1-distill-qwen-14b/transformers/default/1"

    # ---- vLLM inference settings ----
    tensor_parallel_size: int  = 2      # Use both H100s in parallel
    gpu_memory_utilization: float = 0.90
    max_model_len: int         = 32768  # Enough for long think chains
    dtype: str                 = "bfloat16"
    enable_prefix_caching: bool = True  # Speeds up repeated prompt prefixes

    # ---- Sampling: primary (thinking) model ----
    temperature_primary: float = 0.6    # Slightly lower for thinking models
    top_p_primary: float       = 0.95
    max_tokens_primary: int    = 16384  # Long to allow deep think chains

    # ---- Sampling: secondary (distilled) model ----
    temperature_secondary: float = 0.8
    top_p_secondary: float       = 0.95
    max_tokens_secondary: int    = 4096

    # ---- Self-consistency: number of samples ----
    # More samples = better majority vote accuracy
    n_samples_primary: int   = 24  # Samples from primary model per problem
    n_samples_secondary: int = 8   # Samples from secondary model per problem

    # ---- TIR: Tool-Integrated Reasoning ----
    max_tir_rounds: int      = 3   # Max code-execute-feedback loops per sample
    code_timeout_sec: int    = 20  # Hard timeout for each code block

    # ---- Answer constraints (AIMO3 uses 5-digit answers) ----
    answer_mod: int          = 100000
    min_answer: int          = 0
    max_answer: int          = 99999

    # ---- Safety ----
    max_time_minutes: int    = 520  # Stay under 9-hour Kaggle limit

cfg = AIMOConfig()
print(f"✅ Config loaded | primary={cfg.model_path_primary.split('/')[-3]}")
# ==========================================
# Cell 4: Load Competition Data
# ==========================================
def load_test_data() -> pd.DataFrame:
    candidates = [
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv",
    ]
    for path in candidates:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"✅ Loaded {len(df)} problems from {path}")
            return df
    raise FileNotFoundError("test.csv not found — check competition data sources")

test_df = load_test_data()

# Auto-detect the problem text column
PROBLEM_COL = next(
    (c for c in ["problem", "question", "prompt"] if c in test_df.columns), None
)
assert PROBLEM_COL is not None, f"No problem column found. Columns: {test_df.columns.tolist()}"
print(f"✅ Problem column: '{PROBLEM_COL}' | Total: {len(test_df)} problems")
# ==========================================
# Cell 5: Safe Code Executor (TIR Core)
# ==========================================
class SafeExecutor:
    """
    Sandboxed Python executor for TIR.
    Uses signal-based timeout (Unix only) for reliable interruption.
    """
    # Whitelist of safe modules available to the sandbox
    _SAFE_BUILTINS = {k: v for k, v in __builtins__.items()
                      if k not in {"open", "exec", "eval", "__import__",
                                   "compile", "input", "memoryview"}} \
        if isinstance(__builtins__, dict) else {}

    def __init__(self, timeout: int = 20):
        self.timeout = timeout
        # Pre-build sandbox globals once; copy on each execution
        self._base_globals: Dict[str, Any] = {
            "__builtins__": self._SAFE_BUILTINS,
            "math": math,
            "np": np,
            "sympy": sp,
        }
        exec("from sympy import *", self._base_globals)
        exec("import math, itertools, functools, collections, fractions, decimal",
             self._base_globals)

    @staticmethod
    def _timeout_handler(signum, frame):
        raise TimeoutError("Code execution timed out")

    def execute(self, code: str) -> Tuple[str, bool]:
        """
        Execute code string in sandbox.
        Returns (output_string, success_bool).
        """
        sandbox = dict(self._base_globals)  # Fresh namespace each call
        stdout_buf = StringIO()
        stderr_buf = StringIO()

        # Register signal-based timeout (works on Linux/Kaggle)
        old_handler = signal.signal(signal.SIGALRM, self._timeout_handler)
        signal.alarm(self.timeout)

        try:
            with contextlib.redirect_stdout(stdout_buf), \
                 contextlib.redirect_stderr(stderr_buf):
                exec(compile(code, "<sandbox>", "exec"), sandbox)
            output = stdout_buf.getvalue().strip()
            err    = stderr_buf.getvalue().strip()
            result = output if output else err
            return result, True
        except TimeoutError:
            return "TIMEOUT: code took too long", False
        except Exception as exc:
            return f"ERROR: {type(exc).__name__}: {exc}", False
        finally:
            signal.alarm(0)  # Cancel alarm
            signal.signal(signal.SIGALRM, old_handler)

executor = SafeExecutor(cfg.code_timeout_sec)
print("✅ SafeExecutor ready")
# ==========================================
# Cell 6: Problem Type Classifier
# ==========================================
# Lightweight keyword-based classifier to route problems to the right prompt style.
# Top AIMO2 teams found that specialized prompts for each problem type boost accuracy.

PROBLEM_TYPES = {
    "combinatorics": ["ways", "choose", "permutation", "combination", "arrange",
                      "count", "how many", "paths", "subset", "committee"],
    "number_theory": ["prime", "divisor", "gcd", "lcm", "congruent", "modulo",
                      "remainder", "integer", "digit", "divisible"],
    "geometry":      ["triangle", "circle", "angle", "area", "perimeter", "polygon",
                      "tangent", "chord", "radius", "inscribed", "circumscribed"],
    "algebra":       ["polynomial", "equation", "function", "root", "sequence",
                      "series", "inequality", "maximum", "minimum", "optimize"],
}

def classify_problem(text: str) -> str:
    """Return dominant problem type or 'general'."""
    text_lower = text.lower()
    scores = {ptype: sum(kw in text_lower for kw in kws)
              for ptype, kws in PROBLEM_TYPES.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "general"
# ==========================================
# Cell 7: Prompt Templates
# ==========================================
# Multiple prompt styles increase diversity → better majority vote coverage.
# Qwen3 thinking mode is activated via enable_thinking=True in apply_chat_template.

# System prompt for deep mathematical reasoning
SYSTEM_PROMPT = """You are an expert mathematician competing at the International Mathematical Olympiad (IMO).
Your task is to solve a challenging math competition problem.

Key instructions:
1. Read the problem carefully and identify the key mathematical structures.
2. Plan your approach before executing it.
3. When you need precise computation, write Python code inside ```python ... ``` blocks.
   The code will be executed and the result returned to you.
4. Verify your answer numerically when possible.
5. Your final answer must be a non-negative integer between 0 and 99999.
6. State your final answer clearly as: \\boxed{<integer>}

Think step by step. Be rigorous."""

# Style variants to maximize diversity among samples
PROMPT_STYLES = [
    # Style 0: Direct IMO framing
    lambda prob, ptype: (
        f"Solve this {ptype} problem from a mathematical olympiad:\n\n{prob}\n\n"
        f"Think carefully. Write Python code for any computation. "
        f"State the final integer answer as \\boxed{{ans}}."
    ),
    # Style 1: Structured approach
    lambda prob, ptype: (
        f"Problem ({ptype}):\n{prob}\n\n"
        f"Approach:\n"
        f"1. Identify the mathematical framework.\n"
        f"2. Develop a solution strategy.\n"
        f"3. Execute with Python if needed.\n"
        f"4. Verify and state \\boxed{{final_answer}}."
    ),
    # Style 2: Computational emphasis
    lambda prob, ptype: (
        f"Math olympiad problem:\n{prob}\n\n"
        f"Use SymPy and Python for exact computation. "
        f"Check with multiple approaches. "
        f"Answer is an integer in [0, 99999]. Format: \\boxed{{n}}."
    ),
    # Style 3: Think-aloud scaffolding
    lambda prob, ptype: (
        f"Competition problem: {prob}\n\n"
        f"Let me solve this systematically.\n"
        f"First, I'll analyze the structure of the problem...\n"
        f"[Write Python code in ```python ``` blocks for calculations.]\n"
        f"Final answer: \\boxed{{integer}}."
    ),
]

def build_chat_messages(problem: str, ptype: str, style_idx: int) -> List[Dict]:
    """Build a chat message list for vLLM chat template application."""
    user_content = PROMPT_STYLES[style_idx % len(PROMPT_STYLES)](problem, ptype)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

print("✅ Prompt templates ready")
# ==========================================
# Cell 8: Answer Extraction
# ==========================================
# Robust multi-strategy extractor with SymPy fallback.

def extract_final_answer(text: str) -> Optional[int]:
    """
    Extract the final integer answer from model output.
    Priority: \boxed{} > 'answer is X' > last number in tail.
    """
    # Strip think block (Qwen3 / DeepSeek R1 style)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)

    # 1) \boxed{...} — most reliable signal
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        # Take the LAST boxed value (often the corrected/final one)
        return _parse_to_int(boxed[-1].strip())

    # 2) "The answer is X" / "= X" patterns near end
    tail = text[-800:]
    patterns = [
        r"(?:answer|result|value)\s*(?:is|=|:)\s*(\d[\d,]*)",
        r"(?:therefore|thus|so|hence)[^\d]*(\d[\d,]+)",
        r"=\s*(\d{1,5})\s*(?:\.|$)",
    ]
    for pat in patterns:
        m = re.search(pat, tail, re.IGNORECASE)
        if m:
            val = _parse_to_int(m.group(1).replace(",", ""))
            if val is not None:
                return val

    # 3) Fallback: last standalone 1-5 digit number in tail
    numbers = re.findall(r"\b(\d{1,5})\b", tail)
    if numbers:
        return _parse_to_int(numbers[-1])

    return None


def _parse_to_int(candidate: str) -> Optional[int]:
    """Parse a string to a valid AIMO answer integer [0, 99999]."""
    candidate = candidate.strip().replace(",", "")
    try:
        # Direct integer
        if re.fullmatch(r"\d+", candidate):
            val = int(candidate) % cfg.answer_mod
            return val if cfg.min_answer <= val <= cfg.max_answer else None

        # Expression like 3*5 or 10**3
        if re.fullmatch(r"[\d\+\-\*\/\^\.]+", candidate):
            val = int(round(eval(candidate))) % cfg.answer_mod  # noqa: S307
            return val if cfg.min_answer <= val <= cfg.max_answer else None

        # SymPy expression (handles fractions, sqrt results, etc.)
        expr = sp.sympify(candidate)
        if expr.is_number:
            val = int(expr) % cfg.answer_mod
            return val if cfg.min_answer <= val <= cfg.max_answer else None

    except Exception:
        pass
    return None


def extract_code_blocks(text: str) -> List[str]:
    """Extract all ```python ... ``` blocks from model output."""
    return re.findall(r"```python\s*(.*?)\s*```", text, re.DOTALL)

print("✅ Answer extractor ready")
# ==========================================
# Cell 9: Load Models via vLLM
# ==========================================
# vLLM provides:
#   - Paged attention for memory efficiency
#   - Continuous batching for high throughput
#   - Prefix caching for repeated system prompts
#   - tensor_parallel_size=2 to span both H100s

def load_vllm_model(model_path: str, tp_size: int = 2) -> Optional[LLM]:
    """Load a model with vLLM. Returns None if path doesn't exist."""
    if not os.path.exists(model_path):
        print(f"⚠️  Model not found at {model_path}, skipping.")
        return None
    print(f"Loading {model_path} with vLLM (tp={tp_size})...")
    llm = LLM(
        model=model_path,
        tensor_parallel_size=tp_size,
        gpu_memory_utilization=cfg.gpu_memory_utilization,
        max_model_len=cfg.max_model_len,
        dtype=cfg.dtype,
        enable_prefix_caching=cfg.enable_prefix_caching,
        trust_remote_code=True,
        seed=SEED,
    )
    print(f"✅ Loaded: {model_path.split('/')[-3] if '/' in model_path else model_path}")
    return llm


print("Loading primary model (Qwen3-32B)...")
primary_llm = load_vllm_model(cfg.model_path_primary, tp_size=cfg.tensor_parallel_size)

print("Loading secondary model (DeepSeek-R1-Distill-14B)...")
secondary_llm = load_vllm_model(cfg.model_path_secondary, tp_size=1)
# ==========================================
# Cell 10: Tokenizer & Chat Template Helpers
# ==========================================
from transformers import AutoTokenizer

def load_tokenizer(model_path: str) -> Optional[Any]:
    if not os.path.exists(model_path):
        return None
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    return tok

primary_tok   = load_tokenizer(cfg.model_path_primary)
secondary_tok = load_tokenizer(cfg.model_path_secondary)

def apply_chat_template(
    tokenizer,
    messages: List[Dict],
    enable_thinking: bool = True,
) -> str:
    """
    Apply the model's chat template.
    For Qwen3, passing enable_thinking=True activates the think-before-answer mode.
    For other models, the kwarg is gracefully ignored.
    """
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        # Model does not support enable_thinking; apply without it
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return text

print("✅ Tokenizers loaded")
# ==========================================
# Cell 11: Batched vLLM Generation
# ==========================================

def batch_generate(
    llm: LLM,
    tokenizer,
    prompts_text: List[str],
    temperature: float,
    top_p: float,
    max_tokens: int,
) -> List[str]:
    """
    Run vLLM batch inference on a list of already-formatted prompt strings.
    Returns list of response strings (same order as input).
    """
    params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        stop=["<|im_end|>", "<|endoftext|>", "<|eot_id|>"],
        skip_special_tokens=True,
    )
    outputs = llm.generate(prompts_text, params)
    return [out.outputs[0].text for out in outputs]


def make_prompts_for_problem(
    problem: str,
    ptype: str,
    tokenizer,
    n_samples: int,
    enable_thinking: bool = True,
) -> List[str]:
    """
    Generate n_samples prompt strings for one problem by cycling through style variants.
    Diversity in prompts → better coverage of the answer space.
    """
    prompts = []
    for i in range(n_samples):
        msgs = build_chat_messages(problem, ptype, style_idx=i)
        prompt_text = apply_chat_template(tokenizer, msgs, enable_thinking=enable_thinking)
        prompts.append(prompt_text)
    return prompts

print("✅ Batch generation helpers ready")
# ==========================================
# Cell 12: Multi-Turn TIR Engine
# ==========================================
# True Tool-Integrated Reasoning:
#   Round 1 → model generates initial response (may include code blocks)
#   Code blocks are executed → output captured
#   Round 2 → model receives code output, continues reasoning
#   Repeat up to max_tir_rounds or until \boxed{} answer found

def run_tir_single(
    llm: LLM,
    tokenizer,
    problem: str,
    ptype: str,
    style_idx: int,
    enable_thinking: bool = True,
) -> Optional[int]:
    """
    Run one TIR trajectory for a single problem.
    Returns the extracted integer answer or None.
    """
    # Build initial conversation
    messages = build_chat_messages(problem, ptype, style_idx=style_idx)
    params = SamplingParams(
        temperature=cfg.temperature_primary if enable_thinking
                    else cfg.temperature_secondary,
        top_p=cfg.top_p_primary if enable_thinking else cfg.top_p_secondary,
        max_tokens=cfg.max_tokens_primary if enable_thinking
                   else cfg.max_tokens_secondary,
        stop=["<|im_end|>", "<|endoftext|>"],
        skip_special_tokens=True,
    )

    full_response = ""

    for tir_round in range(cfg.max_tir_rounds):
        prompt_text = apply_chat_template(tokenizer, messages,
                                          enable_thinking=enable_thinking)
        outputs = llm.generate([prompt_text], params)
        response = outputs[0].outputs[0].text

        # Accumulate full response for answer extraction later
        full_response += response

        # Check if we already have a boxed answer — stop early
        if re.search(r"\\boxed\{", response):
            break

        # Extract and execute code blocks
        code_blocks = extract_code_blocks(response)
        if not code_blocks:
            break  # No code to execute; stop TIR loop

        # Execute the last code block (most recent computation)
        exec_results = []
        for code in code_blocks[-cfg.max_tir_rounds:]:
            out, success = executor.execute(code.strip())
            status = "Output" if success else "Error"
            exec_results.append(f"```\n{status}:\n{out}\n```")

        code_output = "\n".join(exec_results)

        # Feed code result back into conversation and continue
        messages.append({"role": "assistant", "content": response})
        messages.append({
            "role": "user",
            "content": (
                f"Code execution results:\n{code_output}\n\n"
                f"Continue your solution using these results. "
                f"State your final answer as \\boxed{{integer}}."
            ),
        })
        # Reduce tokens for follow-up turns (answer is nearby)
        params = SamplingParams(
            temperature=max(0.3, params.temperature - 0.1),  # Cool down each round
            top_p=params.top_p,
            max_tokens=min(4096, cfg.max_tokens_primary // 2),
            stop=["<|im_end|>", "<|endoftext|>"],
            skip_special_tokens=True,
        )

    return extract_final_answer(full_response)

print("✅ TIR engine ready")
# ==========================================
# Cell 13: Self-Consistency Ensemble
# ==========================================

def weighted_majority_vote(
    answers: List[Optional[int]],
    weights: Optional[List[float]] = None,
) -> Tuple[int, float, Dict[int, float]]:
    """
    Weighted majority vote over candidate answers.
    Returns (best_answer, confidence, score_distribution).
    """
    valid_pairs = [(a, w) for a, w in zip(answers, weights or [1.0] * len(answers))
                   if a is not None]

    if not valid_pairs:
        return 0, 0.0, {}

    score_map: Dict[int, float] = {}
    for ans, w in valid_pairs:
        score_map[ans] = score_map.get(ans, 0.0) + w

    total_weight = sum(w for _, w in valid_pairs)
    best_ans = max(score_map, key=score_map.get)
    confidence = score_map[best_ans] / total_weight if total_weight > 0 else 0.0

    return best_ans, confidence, score_map

print("✅ Ensemble voting ready")
# ==========================================
# Cell 14: Full Problem Solver
# ==========================================

def solve_problem(problem_text: str, problem_id: Any) -> Dict:
    """
    Full pipeline for one problem:
      1) Classify problem type
      2) Generate n_samples_primary responses from Qwen3-32B (thinking mode)
      3) Optionally generate n_samples_secondary from DeepSeek-R1-Distill
      4) Run TIR for low-confidence cases
      5) Weighted majority vote
    """
    t0 = time.time()
    ptype = classify_problem(problem_text)
    print(f"    Type: {ptype}")

    all_answers:  List[Optional[int]] = []
    all_weights:  List[float]         = []

    # ---- Phase 1: Batch generation from primary model (Qwen3-32B) ----
    if primary_llm is not None and primary_tok is not None:
        prompts = make_prompts_for_problem(
            problem_text, ptype, primary_tok,
            n_samples=cfg.n_samples_primary, enable_thinking=True
        )
        responses = batch_generate(
            primary_llm, primary_tok, prompts,
            temperature=cfg.temperature_primary,
            top_p=cfg.top_p_primary,
            max_tokens=cfg.max_tokens_primary,
        )
        for resp in responses:
            ans = extract_final_answer(resp)
            all_answers.append(ans)
            all_weights.append(2.0)  # Primary model gets higher weight

    # ---- Phase 2: Batch generation from secondary model ----
    if secondary_llm is not None and secondary_tok is not None:
        prompts_sec = make_prompts_for_problem(
            problem_text, ptype, secondary_tok,
            n_samples=cfg.n_samples_secondary, enable_thinking=True
        )
        responses_sec = batch_generate(
            secondary_llm, secondary_tok, prompts_sec,
            temperature=cfg.temperature_secondary,
            top_p=cfg.top_p_secondary,
            max_tokens=cfg.max_tokens_secondary,
        )
        for resp in responses_sec:
            ans = extract_final_answer(resp)
            all_answers.append(ans)
            all_weights.append(1.0)

    # ---- Phase 3: Intermediate vote to check confidence ----
    best_ans, confidence, score_map = weighted_majority_vote(all_answers, all_weights)
    print(f"    After batch gen: ans={best_ans}, conf={confidence:.1%}, "
          f"valid={sum(1 for a in all_answers if a is not None)}/{len(all_answers)}")

    # ---- Phase 4: TIR refinement for low-confidence problems ----
    # If confidence < 50%, or no valid answers, run TIR for deeper reasoning.
    if confidence < 0.50 and primary_llm is not None and primary_tok is not None:
        print("    Low confidence — running TIR refinement...")
        for tir_style in range(4):  # Try 4 different TIR trajectories
            tir_ans = run_tir_single(
                primary_llm, primary_tok, problem_text, ptype,
                style_idx=tir_style, enable_thinking=True
            )
            if tir_ans is not None:
                all_answers.append(tir_ans)
                all_weights.append(3.0)  # TIR answers get highest weight (most compute)
        # Re-vote with TIR results
        best_ans, confidence, score_map = weighted_majority_vote(all_answers, all_weights)
        print(f"    After TIR: ans={best_ans}, conf={confidence:.1%}")

    # ---- Final sanity check ----
    best_ans = int(best_ans) % cfg.answer_mod
    best_ans = max(cfg.min_answer, min(cfg.max_answer, best_ans))

    elapsed = time.time() - t0
    return {
        "id":               problem_id,
        "answer":           best_ans,
        "confidence":       round(confidence, 4),
        "num_valid":        sum(1 for a in all_answers if a is not None),
        "num_total":        len(all_answers),
        "problem_type":     ptype,
        "elapsed_seconds":  round(elapsed, 1),
        "score_map":        score_map,
    }

print("✅ Problem solver ready")
# ==========================================
# Cell 15: Main Inference Loop
# ==========================================
print("\n" + "=" * 65)
print("🚀 AIMO3 Gold Medal Pipeline — Starting")
print(f"   Problems: {len(test_df)}")
print(f"   Primary samples: {cfg.n_samples_primary}  |  "
      f"Secondary samples: {cfg.n_samples_secondary}")
print("=" * 65 + "\n")

results: List[Dict] = []
global_start = time.time()

for idx, row in test_df.iterrows():
    problem_id   = row["id"]
    problem_text = row[PROBLEM_COL]

    elapsed_minutes = (time.time() - global_start) / 60
    remaining_minutes = cfg.max_time_minutes - elapsed_minutes

    print(f"[{idx + 1}/{len(test_df)}] ID={problem_id} "
          f"| Elapsed={elapsed_minutes:.1f}m | Remaining={remaining_minutes:.1f}m")

    # Hard time cut-off: leave 10 min buffer for submission creation
    if remaining_minutes < 10:
        print("⏰ Time limit approaching — stopping inference, filling defaults")
        for remaining_row in test_df.iloc[idx:].itertuples():
            results.append({
                "id":              remaining_row.id,
                "answer":          0,
                "confidence":      0.0,
                "num_valid":       0,
                "num_total":       0,
                "problem_type":    "unknown",
                "elapsed_seconds": 0,
                "score_map":       {},
            })
        break

    try:
        result = solve_problem(problem_text, problem_id)
        results.append(result)
        print(f"    → Answer={result['answer']}  "
              f"Conf={result['confidence']:.1%}  "
              f"Valid={result['num_valid']}/{result['num_total']}  "
              f"Time={result['elapsed_seconds']}s")
    except Exception as exc:
        print(f"    → EXCEPTION: {exc}")
        results.append({
            "id":              problem_id,
            "answer":          0,
            "confidence":      0.0,
            "num_valid":       0,
            "num_total":       0,
            "problem_type":    "error",
            "elapsed_seconds": 0,
            "score_map":       {},
        })
# ==========================================
# Cell 16: Build & Save Submission
# ==========================================
results_df = pd.DataFrame(results)

# Merge with original IDs to ensure every problem has an answer
submission_df = test_df[["id"]].copy()
submission_df = submission_df.merge(
    results_df[["id", "answer"]], on="id", how="left"
)
submission_df["answer"] = (
    submission_df["answer"]
    .fillna(0)
    .astype(int)
    .clip(cfg.min_answer, cfg.max_answer)
)

output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(output_path, index=False)

total_runtime = (time.time() - global_start) / 60
print("\n" + "=" * 65)
print("🎉 SUBMISSION READY")
print(f"   Saved to: {output_path}")
print(f"   Total runtime: {total_runtime:.1f} minutes")
print("\n📊 Submission preview:")
print(submission_df.head(10).to_string(index=False))

print("\n📈 Confidence distribution:")
print(results_df["confidence"].describe().round(3).to_string())

valid_mask = results_df["num_valid"] > 0
print(f"\n✅ Problems with valid answer: "
      f"{valid_mask.sum()}/{len(results_df)} "
      f"({100 * valid_mask.mean():.1f}%)")

high_conf_mask = results_df["confidence"] >= 0.5
print(f"🔒 High confidence (≥50%): "
      f"{high_conf_mask.sum()}/{len(results_df)} "
      f"({100 * high_conf_mask.mean():.1f}%)")

print("\n" + "=" * 65)
print("✅ AIMO3 Gold Medal Pipeline Complete!")

config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

ModuleNotFoundError: No module named 'transformers.models.audioflamingo3'